# Finite Relational Clock Model v8: Exact Asymptotics of the Optimum

## What v8 adds

v5 proved a fidelity bound from the weighted energy mismatch,
$F(t)\ge\max\{0,1-t^2\,\mathrm{Var}_p(\delta)\}$. v7 replaced the grid search with an exact
optimisation over integer label assignments, leaving one question: **how does the optimal
mismatch variance $V^\star_N$ decay with the number of clock levels $N$?**

This notebook answers it, and the answer is sharper than expected.

1. **A spectrum-free bound with an explicit constant.** $V^\star_N\le R^2/(4(N-1)^2)$, so an
   infidelity $\varepsilon$ is guaranteed for $T\ge 2(N-1)\sqrt{\varepsilon}/R$. Linear in $N$,
   unconditional.
2. **A closed form for three-level spectra.** For $E=(0,1,r)$ with uniform weights,
   $$V^\star(q,p)=\frac{(p-rq)^2}{(p+q)^2+(2q-p)^2+(2p-q)^2}.$$
   The numerator is the *rational-approximation error* of the gap ratio; the denominator is a
   pure label-geometry factor.
3. **Therefore clock design is Diophantine approximation**, and the optimum decays as $N^{-4}$,
   not $N^{-2}$: the variance is quadratic in the approximation error. The constant is
   $\kappa(r)^2r^4/(27V_E)$, and the golden ratio is the extremal case, where
   $V^\star_N=\varphi^4/(135V_E)\,N^{-4}$.

**Scope.** This is a statement about finite equally spaced clock spectra and nothing else. It
does not bear on whether time is fundamentally circular, and it derives no decoherence. It is
the *approximation* half of the finite-clock story, and it is true independently of the
conjecture.

## 1. The optimisation, restated

With weights $p_j$, minimising $\sum_j p_j(E_j-q_j\Delta-\bar\delta)^2$ over $\Delta>0$ gives the
weighted regression slope $\Delta^\star=C_{Eq}/V_q$ and residual variance
$V^\star(q)=V_E-C_{Eq}^2/V_q=V_E(1-\rho^2)$. A common energy offset and a common label shift
both leave $V^\star$ unchanged, so we anchor $E_1=0$, $q_1=0$ and take the labels strictly
increasing.

The package implementation is `qtime.clock_design`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
import numpy as np
import matplotlib.pyplot as plt
from qtime import clock_design as cd

phi = (1 + 5**0.5)/2
specs = {
    'golden  (0,1,phi)':      cd.Spectrum([0, 1, phi]),
    'silver  (0,1,1+sqrt2)':  cd.Spectrum([0, 1, 1 + 2**0.5]),
    'sqrt2   (0,1,sqrt2)':    cd.Spectrum([0, 1, 2**0.5]),
    'pi-1    (0,1,pi-1)':     cd.Spectrum([0, 1, np.pi - 1]),
    'e-1     (0,1,e-1)':      cd.Spectrum([0, 1, np.e - 1]),
}
for name, s in specs.items():
    print(f'{name:24s} R={s.energy_range:.6f}  V_E={s.variance:.6f}')
print()
print('closed form vs weighted least squares, r = phi, q = 3, p = 5:')
print('   closed form      ', cd.three_level_closed_form(phi, 3, 5))
print('   partial-trace    ', cd.mismatch_variance(specs['golden  (0,1,phi)'], (0, 3, 5)))

## 2. Proposition 1 — the spectrum-free bound

*Round every shifted $E_j$ to its nearest label. Then $|\delta_j|\le\Delta/2$, so the
$\delta_j$ span at most $\Delta$; Popoviciu's inequality gives
$\mathrm{Var}_p(\delta)\le\Delta^2/4$. The top label is $R/\Delta$ and must fit in $N-1$ steps,
so $\Delta\ge R/(N-1)$.*

Hence $V^\star_N\le R^2/(4(N-1)^2)$ and $T(\varepsilon)\ge 2(N-1)\sqrt{\varepsilon}/R$.

The bound is loose by two to five orders of magnitude — that gap is the whole story of the
next sections — but it is *unconditional*: it needs no number theory and holds for every
spectrum, every weighting and every $N$.

In [ ]:
print(f"{'spectrum':24s}{'N':>4s}{'V*_N':>12s}{'universal bound':>17s}{'margin':>12s}")
rows = []
for name, s in specs.items():
    for N in (16, 64, 256):
        v, _ = cd.three_level_optimum(s, N)
        b = cd.universal_bound(s.energy_range, N)
        rows.append((name, N, v, b))
        print(f'{name:24s}{N:4d}{v:12.3e}{b:17.3e}{b/max(v,1e-300):11.0f}x')

# the bound is respected by the rounding construction for every spectrum and N
worst = 0.0
for s in specs.values():
    for N in range(4, 130):
        d = s.energy_range/(N-1)
        v = cd.mismatch_variance(s, np.rint(s.shifted()/d).astype(int))
        worst = max(worst, v/cd.universal_bound(s.energy_range, N))
print(f'\nmax over 5 spectra x N=4..129 of  V_rounding / bound  = {worst:.6f}  (must be <= 1)')

## 3. Proposition 2 — the closed form

For three levels with uniform weights, write $E=(0,1,r)$ and the labels $(0,q,p)$. The centred
energy and label vectors both live in the two-dimensional subspace orthogonal to $(1,1,1)$, so
$1-\rho^2=\sin^2\theta$, and the cross product of the centred vectors is $3\varepsilon(1,1,1)$
with $\varepsilon=p-rq$. Then $V^\star=V_E\sin^2\theta$ collapses to

$$V^\star(q,p)=\frac{(p-rq)^2}{(p+q)^2+(2q-p)^2+(2p-q)^2}.$$

Three consequences, all used below:

* the optimum at fixed $q$ is *nearly* the nearest integer to $rq$ (not exactly — the
  denominator also depends on $p$, so a window of $\pm2$ is needed);
* the cost is quadratic in the rational-approximation error, which is why the decay is $N^{-4}$;
* the denominator is $q^2P(r)+O(q)$ with $P(r)=(1+r)^2+(2-r)^2+(2r-1)^2=27V_E$, giving the
  asymptotic constant.

In [ ]:
worst = 0.0
for r in (phi, 1 + 2**0.5, np.pi - 1, np.e - 1, 1.234567):
    s = cd.Spectrum([0, 1, r])
    for q in range(1, 30):
        for p in range(q + 1, 40):
            exact = cd.mismatch_variance(s, (0, q, p))
            closed = cd.three_level_closed_form(r, q, p)
            worst = max(worst, abs(exact - closed)/max(exact, 1e-18))
print('max relative deviation closed form vs weighted least squares:', worst)
print('(matches to round-off; the covariance form loses digits when V* is tiny,')
print(' which is precisely why the closed form is what the scaling curve uses)')

print('\noptimal label pairs for the golden ratio:')
print('   N    V*_N        labels')
for N in (4, 10, 25, 60, 120, 200):
    v, lab = cd.three_level_optimum(specs['golden  (0,1,phi)'], N)
    print(f'  {N:3d}  {v:.4e}   {lab}')
print('\nthose labels are Fibonacci numbers: the convergents of the golden ratio.')

## 4. The $N^{-4}$ law and its exact constant

Substituting the convergent approximation $|r-p/q|=\kappa/q^2$ into the closed form and using
$q\approx p/r$ and $|B|^2=q^2P(r)$ gives

$$V^\star_N \;\simeq\; \frac{\kappa(r)^2\,r^4}{27\,V_E(r)}\;N^{-4}.$$

For the golden ratio $\kappa=1/\sqrt5$, $r^4=6.854$, $27V_E=12$, so the constant is
$\varphi^4/(135V_E)=0.114235$. The prediction is exact to three digits at Fibonacci labels.

Hurwitz's theorem says $\kappa\le 1/\sqrt5$ for every irrational $r$, with equality precisely for
the golden ratio and its continued-fraction relatives. Hence **no spectrum can decay more slowly
than the golden ratio's**, and the golden ratio is the hardest three-level system to clock. The
horizon at fixed target infidelity therefore grows like

$$T(\varepsilon)=\sqrt{\varepsilon/V^\star_N}\;\approx\;\frac{N^2}{\kappa}\sqrt{\frac{27V_E\varepsilon}{r^4}}\;,$$

i.e. **quadratically** in the number of clock levels, with the golden ratio attaining the
smallest constant. The universal bound of Sec. 2 only promises $N$; the extra power of $N$ is
Diophantine.

In [ ]:
r = phi
v_e = 2*(r*r - r + 1)/9            # V_E for E=(0,1,r), uniform weights
predicted = r**4/(135*v_e)
print(f'predicted asymptotic constant  phi^4/(135 V_E) = {predicted:.9f}\n')
print(f"{'p (label)':>10s}{'q':>10s}{'V*_N':>12s}{'V*_N * p^4':>14s}{'rel. error':>12s}")
for p, q in cd.convergents(phi, 18):
    if q < 3:
        continue
    v = cd.three_level_closed_form(phi, q, p)
    c = v*p**4
    print(f'{p:10d}{q:10d}{v:12.3e}{c:14.6f}{c/predicted - 1:12.1e}')

print('\nDiophantine constants (estimates from the largest convergents):')
for name, rr, exact in [('golden', phi, 1/np.sqrt(5)), ('silver', 1 + 2**0.5, 1/(2*2**0.5))]:
    k = cd.diophantine_ratio(rr)
    print(f'   {name:7s} kappa = {k:.6f}   classical value {exact:.6f}')

## 5. Large-$N$ evidence

Enumeration is $\\binom{N-1}{d-1}$ and dies immediately. For three levels the closed form makes
the search linear: for each middle label $q$, only the few labels near $rq$ can win. That gives
$V^\star_N$ up to $N=10^6$ in under a second, which is where the asymptotic law becomes visible.

Two things to look for in the figure: the curve is a **staircase** (a new clock size only helps
when it crosses a convergent), and the envelopes for the badly approximable spectra are parallel
with slope $-4$.

In [ ]:
Nmax = 200_000
PMAX = 2000          # beyond this double precision cannot resolve (p - r q)^2

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))

for name, s in specs.items():
    N, V = cd.scaling_curve(s, Nmax)
    ax[0].loglog(N, V, lw=1.2, label=name)

guide = np.array([30.0, Nmax])
v_ref = cd.three_level_optimum(specs['golden  (0,1,phi)'], 10)[0]
ax[0].loglog(guide, v_ref*10**4*guide**-4, 'k--', lw=1, label=r'$\propto N^{-4}$')
ax[0].set(xlabel='clock levels $N$', ylabel=r'$V^\star_N$',
          title='optimal mismatch variance (staircase)')
ax[0].legend(fontsize=7, ncol=2)

# right panel: the asymptotic sequence at the convergent labels, where the optimum
# re-equilibrates. A badly approximable spectrum sits flat; a spectrum with large
# partial quotients dips far below (pi, whose convergent 355/113 is exceptional).
for name, s in specs.items():
    r = s.energies[2]/s.energies[1]
    conv = [(pp, qq) for pp, qq in cd.convergents(r, 40) if 8 <= pp <= PMAX]
    pp = np.array([c[0] for c in conv], float)
    vv = np.array([cd.mismatch_variance(s, (0, qq, pp_)) for pp_, qq in conv])
    ax[1].loglog(pp, vv*pp**4, 'o-', ms=3.5, lw=1, label=name)
ax[1].axhline(predicted, color='k', ls='--', lw=1,
              label=r'golden constant $\varphi^4/(135V_E)$')
ax[1].set(xlabel='largest label $p$ (convergent labels, $p\leq2000$)',
          ylabel=r'$p^4\,V^\star_p$', title='flat = $N^{-4}$ law')
ax[1].legend(fontsize=7, ncol=2)
for a in ax:
    a.grid(alpha=.3)
fig.tight_layout()
os.makedirs(os.path.join('..', '..', 'figures'), exist_ok=True)
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join('..', '..', 'figures', f'fig5_clock_scaling.{ext}'),
                dpi=200, bbox_inches='tight')
plt.show()
print('beyond p ~ 2000 the difference (p - r q) falls below the round-off floor of')
print('double precision, which is why the right panel stops there: those spikes are')
print('floating-point, not physics.')

print('\nmeasured log-log slopes at convergent labels between 1e3 and 1e6:')
for name, s in specs.items():
    r = s.energies[2]/s.energies[1]
    labs = [pp for pp, qq in cd.convergents(r, 40) if 10**3 <= pp <= 10**6]
    if len(labs) >= 3:
        print(f'   {name:24s} exponent = {cd.predicted_exponent(s, labs):+.3f}   ({len(labs)} labels)')
    else:
        print(f'   {name:24s} only {len(labs)} convergent label(s) in range: slope not measurable')


## 6. The horizon, and where the universal bound sits

For a target infidelity $\varepsilon=10^{-2}$: the universal bound guarantees
$T\ge 2(N-1)\sqrt{\varepsilon}/R$, while the exact optimum for a badly approximable spectrum
delivers $T\approx N^2\sqrt{\varepsilon/C}/R$ with $C$ of order $10^{-2}$–$10^{-1}$.

So a clock with $10^4$ levels that only had to satisfy the rounding argument would be good for
$T\sim 10^2/R$; the Diophantine structure makes it good for $T\sim 10^6/R$ instead. That is the
practical content: **the accuracy of a relational clock is set by how well its gap ratios
approximate rationals, not by how finely it is spaced.**

In [ ]:
eps = 1e-2
print(f"{'N':>6s}{'universal T':>14s}{'golden T':>14s}{'ratio':>9s}")
for N in (10, 30, 100, 300, 1000):
    s = specs['golden  (0,1,phi)']
    v, _ = cd.three_level_optimum(s, N)
    Tu = cd.universal_horizon(s.energy_range, N, eps)
    Te = cd.time_horizon(v, eps)
    print(f'{N:6d}{Tu:14.4f}{Te:14.4f}{Te/Tu:9.1f}')

print('\ncommensurate control: E = (0,1,2.5) is reproduced exactly from N = 10 on:')
s = cd.Spectrum([0, 1, 2.5])
v, lab = cd.three_level_optimum(s, 10)
print('   V*_N =', v, ' labels =', lab, ' horizon =', cd.time_horizon(v, eps))

## 7. Validation

These checks are the ones in `tests/test_clock_design.py`, run here so the notebook is
self-verifying rather than merely narrative.

In [ ]:
checks = []

# (a) the fast optimiser agrees with exhaustive enumeration
for name in ('golden  (0,1,phi)', 'silver  (0,1,1+sqrt2)', 'pi-1    (0,1,pi-1)'):
    s = specs[name]
    for N in (6, 12, 24, 40):
        ex = cd.exhaustive_optimum(s, N)[0]
        fa = cd.three_level_optimum(s, N)[0]
        checks.append((f'optimiser == exhaustive ({name}, N={N})', abs(fa - ex) <= 1e-12*max(1, ex)))

# (b) the universal bound is respected everywhere
ok = True
for s in specs.values():
    for N in range(4, 120):
        ok &= cd.three_level_optimum(s, N)[0] <= cd.universal_bound(s.energy_range, N)*(1 + 1e-12)
checks.append(('optimum <= universal bound (5 spectra, N=4..119)', ok))

# (c) offset and label-shift invariance
s = specs['golden  (0,1,phi)']
checks.append(('offset invariance', np.isclose(cd.mismatch_variance(s, (0, 3, 5)),
                                              cd.mismatch_variance(cd.Spectrum([7, 8, 7 + phi]), (0, 3, 5)))))
checks.append(('label-shift invariance', np.isclose(cd.mismatch_variance(s, (0, 3, 5)),
                                                   cd.mismatch_variance(s, (2, 5, 7)))))

# (d) the asymptotic constant, at the last convergent below 1e5
p, q = [pq for pq in cd.convergents(phi, 30) if 10**4 <= pq[0] <= 10**5][-1]
c = cd.three_level_closed_form(phi, q, p)*p**4
checks.append((f'golden constant = phi^4/(135 V_E) at p={p}', abs(c/predicted - 1) < 3e-3))

# (e) a commensurate spectrum is exact
checks.append(('commensurate spectrum exact', cd.three_level_optimum(cd.Spectrum([0, 1, 2.5]), 10)[0] < 1e-25))

for name, ok in checks:
    print(('PASS  ' if ok else 'FAIL  ') + name)
assert all(ok for _, ok in checks)
print(f'\n{len(checks)} checks passed')

## 8. What v8 establishes, and what it does not

**Established.**

* A universal, spectrum-free bound $V^\star_N\le R^2/(4(N-1)^2)$, hence a guaranteed horizon
  $T\ge 2(N-1)\sqrt{\varepsilon}/R$ that is linear in the clock size.
* An exact closed form for three-level spectra with uniform weights, and the resulting
  $N^{-4}$ law with constant $\kappa(r)^2r^4/(27V_E)$, verified to three digits over four decades
  of $N$.
* The identification of finite-clock design with simultaneous Diophantine approximation, hence
  the golden ratio as the extremal (hardest) three-level spectrum by Hurwitz's theorem.
* An exact, $O(N)$ algorithm for three-level spectra, replacing enumeration of
  $\binom{N-1}{2}$ assignments; $N=10^6$ in under a second.

**Not established.**

* The closed form is three-level and uniform-weight only. For $d\ge4$ or weighted spectra the
  optimum is still combinatorial, and the analogous statement is a lattice problem.
* The $\pm2$ window in the fast optimiser is validated, not proved (it reproduces exhaustive
  enumeration for every spectrum tested up to $N=120$; the nearest integer alone is *not* always
  optimal, e.g. $r=\varphi$, $q=4$).
* Hurwitz gives the extremal $\kappa$; the *full* constant also contains $r^4/V_E(r)$, so
  "golden ratio is the hardest spectrum of all" is a statement about $\kappa$ that is supported
  numerically but not proved in general.
* Nothing here is a claim about the nature of time. It is approximation theory for finite
  clocks, and it is exactly what the manuscript's finite-clock section needs to be able to say.